# Stage 1 and Stage 2. Quality Control and Illumination Normalization

Notebook ini menjalankan quality control pada citra region of interest lalu menstabilkan pencahayaan memakai CLAHE pada channel V dan menandai pixel valid sebelum ekstraksi fitur. Quality control diposisikan sebagai gerbang untuk capture di lapangan, sedangkan pada dataset kurasi metriknya hanya dilaporkan tanpa membuang data latih berlabel.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from configs import paths
from src import data, qc, preprocess

manifest = data.build_manifest(save=False)
print("manifest", manifest.shape)

## Quality Control Metrics

Setiap citra dinilai berdasarkan ketajaman, kecerahan, fraksi glare, dan luas region of interest. Ambang default dikalibrasi untuk capture di lapangan. Sebagian kecil citra CP-AnemiC ditandai karena glare atau eksposur berlebih, sedangkan Eyes-Defy lolos penuh.

In [ ]:
qc_report = qc.run_quality_check(manifest)
print("overall pass rate", round(100 * qc_report["passed"].mean(), 1), "percent")
print()
print("pass rate per dataset:")
print(qc_report.groupby("dataset")["passed"].mean().mul(100).round(1).to_string())
print()
reasons = qc_report.loc[~qc_report["passed"], "reasons"].str.split(",").explode()
print("failure reasons:")
print(reasons[reasons != ""].value_counts().to_string())

## Metric Distributions

Distribusi metrik per dataset membantu memahami perbedaan karakteristik citra dan mengkalibrasi ambang. Strip CP-AnemiC berukuran kecil sehingga variance of Laplacian cenderung lebih rendah dibanding cutout Eyes-Defy.

In [ ]:
columns = ["laplacian_variance", "brightness", "glare_fraction"]
titles = ["Sharpness (Laplacian variance)", "Brightness", "Glare fraction"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, column, title in zip(axes, columns, titles):
    for name, group in qc_report.groupby("dataset"):
        ax.hist(group[column], bins=30, alpha=0.6, label=name)
    ax.set_title(title)
    ax.set_ylabel("Count")
    ax.legend()
plt.tight_layout()
plt.show()

## Illumination Normalization

CLAHE pada channel V meningkatkan kontras lokal dan meredam pencahayaan tidak merata tanpa mengubah informasi warna. Contoh diambil dari tiga varian mask, yaitu strip CP-AnemiC, cutout alpha Eyes-Defy India, dan cutout latar putih Eyes-Defy Italy.

In [ ]:
samples = ["cp_Image_001", "india_001", "italy_001"]
fig, axes = plt.subplots(len(samples), 2, figsize=(9, 3 * len(samples)))
for i, uid in enumerate(samples):
    row = manifest[manifest["uid"] == uid].iloc[0]
    original, _ = qc.load_roi(row["roi_path"])
    result = preprocess.normalize_roi(row["roi_path"])
    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f"{uid} original")
    axes[i, 1].imshow(result["rgb"])
    axes[i, 1].set_title(f"{uid} CLAHE normalized")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()
plt.show()

## Valid Pixel Filtering

Setelah normalisasi, pixel yang terlalu gelap atau terlalu terang dibuang agar ekstraksi fitur hanya memakai jaringan konjungtiva yang representatif.

In [ ]:
row = manifest[manifest["uid"] == "india_001"].iloc[0]
result = preprocess.normalize_roi(row["roi_path"])
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(result["rgb"])
axes[0].set_title("Normalized ROI")
axes[1].imshow(result["valid_mask"], cmap="gray")
axes[1].set_title("Valid pixel mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Save QC Report

Laporan quality control disimpan ke folder outputs sebagai referensi kalibrasi dan audit kualitas citra.

In [ ]:
output_path = paths.OUTPUTS / "qc_report.csv"
qc_report.to_csv(output_path, index=False)
print("qc report saved to", output_path)